# Banka Müşteri Churn Tahmini — Keşifsel Veri Analizi

Veri setinin genel yapısı, değişken dağılımları ve churn ile ilişkili örüntüler incelenmektedir.

## 1. Kütüphaneler ve Veri Yükleme

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

churn = pd.read_csv('Churn.csv')
df = churn.copy()

## 2. Keşifsel Veri Analizi

In [ ]:
df.describe().T

In [ ]:
df.info()

In [ ]:
numeric_columns = df.select_dtypes(include=[np.number])
print('Sayısal sütunlar:')
print(numeric_columns)

In [ ]:
def plot_histogram(df, column_name):
    plt.figure(figsize=(5, 3))
    sns.histplot(df[column_name], kde=True)
    plt.title(f'Distribution of {column_name}')
    col_mean = df[column_name].mean()
    col_median = df[column_name].median()
    plt.axvline(col_mean, color='red', linestyle='--', label='Mean')
    plt.axvline(col_median, color='green', linestyle='-', label='Median')
    plt.legend()
    plt.show()

def plot_boxplot(df, column_name):
    plt.figure(figsize=(5, 3))
    sns.boxplot(y=df[column_name])
    plt.title(f'Box Plot of {column_name}')
    plt.ylabel(column_name)
    plt.show()

### 2.1 Korelasyon Matrisi

`Balance` ile `NumOfProducts` arasında yaklaşık -0.3 düzeyinde negatif korelasyon gözlemlenmiştir. 0.5 eşiğinin altında kaldığından bu ilişki ihmal edilmiştir.

In [ ]:
plt.figure(figsize=(12, 10))
numeric_df = df.select_dtypes(include=['float64', 'int64'])
cor = numeric_df.corr()
sns.heatmap(cor, annot=True, cmap=plt.cm.Reds)
plt.title('Korelasyon Matrisi')
plt.show()

### 2.2 Hedef Değişken: Exited

Veri setinde sınıf dağılımı %20 churn / %80 kalmış şeklindedir. Bu dengesizlik ilerleyen aşamada SMOTE ile giderilecektir.

In [ ]:
df['Exited'].value_counts()

In [ ]:
%matplotlib inline
print(f'Churn oranı: {2037/9999:.2%}')
sns.countplot(x='Exited', data=df, label='Count', palette='viridis')
plt.title('Churn Dağılımı (0: Kaldı, 1: Ayrıldı)')
plt.show()

### 2.3 Değişken Analizleri

#### CreditScore

Kredi skoru 405'in altında olan tüm müşterilerin churn olduğu görülmüştür (20 gözlem). Bu gözlem doğrultusunda `smallerthan405` ikili değişkeni türetilmiştir.

In [ ]:
df['CreditScore'].describe().T

In [ ]:
sns.boxplot(df['CreditScore'], orient='h')
plt.title('CreditScore Boxplot')
plt.show()

In [ ]:
# Kredi skoru 400'ün altında olanların tamamı churn olmuş
df[df['CreditScore'] < 405].apply({'Exited': 'value_counts'})

In [ ]:
plot_histogram(df, 'CreditScore')

#### Age

Yaş dağılımında üst değerler outlier izlenimi verse de gerçek müşteri verisi olduğundan veri setinde bırakılmış, gruplandırma yoluyla dönüştürülmüştür.

In [ ]:
df['Age'].describe()

In [ ]:
sns.boxplot(df['Age'], orient='h')
plt.title('Age Boxplot')
plt.show()

In [ ]:
plot_histogram(df, 'Age')

#### Tenure

0 ve 10 yıllık müşteri sayısı diğer gruplara kıyasla daha azdır; dağılım genel olarak düzgündür.

In [ ]:
df['Tenure'].value_counts()

In [ ]:
plot_histogram(df, 'Tenure')
plot_boxplot(df, 'Tenure')

#### Balance

Bakiyesi sıfır olan müşterilerin churn oranı, bakiyesi sıfır olmayanlara kıyasla daha düşük çıkmıştır. Bu gözlem `Balance0` ikili değişkeninin türetilmesine zemin hazırlamıştır.

In [ ]:
df['Balance'].describe()

In [ ]:
sns.boxplot(df['Balance'], orient='h')
plt.show()

In [ ]:
print('Balance > 100.000 olanlar:', df[df['Balance'] > 100000]['Exited'].mean())
print('Balance = 0 olanlar:', df[df['Balance'] == 0]['Exited'].mean())
print('Balance != 0 olanlar:', df[df['Balance'] != 0]['Exited'].mean())

In [ ]:
plot_histogram(df, 'Balance')

#### NumOfProducts

3 veya 4 ürüne sahip müşterilerin churn oranı son derece yüksektir (sırasıyla %82 ve %100). Bu bilgi `NOP*` değişkeninin oluşturulmasında kullanılmıştır.

In [ ]:
df['NumOfProducts'].value_counts()

In [ ]:
for n in [1, 2, 3, 4]:
    rate = df[df['NumOfProducts'] == n]['Exited'].mean()
    print(f'NumOfProducts={n}: Churn oranı = {rate:.2%}')

#### HasCrCard, Geography, Gender

Kredi kartı sahipliğinin churn üzerinde anlamlı bir etkisi bulunmamıştır. Alman müşterilerin terk oranı (%32) Fransız (%16) ve İspanyol (%17) müşterilere kıyasla belirgin biçimde yüksektir.

In [ ]:
print('HasCrCard - Exited korelasyonu:', df['HasCrCard'].corr(df['Exited']))
sns.countplot(x='HasCrCard', data=df, palette='viridis')
plt.title('Kredi Kartı Sahipliği')
plt.show()

In [ ]:
gender_counts = df['Gender'].value_counts()
plt.figure(figsize=(7, 7))
plt.pie(gender_counts, labels=gender_counts.index, autopct='%1.1f%%', colors=['lightcoral', 'lightblue'])
plt.title('Cinsiyet Dağılımı')
plt.show()